
# 06 · FC-STEM (cepstral) — 약한 신호·혼합 비정질상 매핑

`RDF(structure factor) 방법`이 신호가 약해 잘 안 될 때의 대안. **EWPC(Exit-Wave Power Cepstrum)**:
$$\mathrm{EWPC}(\mathbf r)=\big|\,\mathcal F^{-1}\{\log I(\mathbf k)\}\,\big|$$
- **log** 이 다이내믹 레인지를 압축 → 약한 산란도 살아남음
- **배경 제거·structure factor 불필요** (실패하던 그 단계를 건너뜀)
- quefrency 축 = 실공간 거리(Å), 피크 = **원자간 거리** (RDF 유사)
- **Fluctuation(정규분산)** $F(R_p)=\langle C_p^2\rangle/\langle C_p\rangle^2-1$ 을 거리 밴드별로 → **혼합상 매핑**

> 참고: Pidaparthy, Ni, Hou, Abraham, Zuo, *Ultramicroscopy* **248** (2023) 113718.
> 켑스트럼은 모듈러스라 **비음수** → NMF도 유효(RDF의 부호 문제 없음).
> 모든 그림 PNG + 그래프 CSV는 각 셀에서 `SAVE_DIR`에 저장됩니다.


## 1) dm4 불러오기 & 설정

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)

DET_BIN    = 2            # 검출기 비닝(메모리/속도). 합성이면 1
Q_UNIT_HINT= "1/nm"     # ★ dm4 단위가 1/nm (0.043888 1/nm/px). 1/A로 두면 거리 10배 틀림
Q_PER_PX   = 0.020        # 초기값. 실데이터는 메타데이터 사용(아래에서 갱신)
N_JOBS     = -2           # 병렬 코어수
EWPC_OFFSET= 1.0          # log(I+offset) — log(0) 방지
WINDOW     = True         # 켑스트럼 전 Hann 창(가장자리 십자 아티팩트 억제)
K          = 3            # 켑스트럼 프로파일 NMF/PCA 성분 수 (elbow 스캔으로 확인 후 조정)
KMAX_SCAN  = 8            # 성분 수 스캔 범위(1..KMAX): elbow/누적분산으로 '진짜 몇 개'인지
CENTER     = None         # (cx,cy) 수동. None이면 무게중심
# 알려진 Li 화합물 원자간 거리(Å, 결정 근사) — 켑스트럼 프로파일에 참고선(비정질은 다소 이동)
CEPSTRAL_REF = {"Li-F 2.01": 2.01, "Li-O 2.00": 2.00, "Li-N 1.94": 1.94,
                "Li-S 2.47": 2.47, "Li-C 2.1": 2.10, "C-C 1.4": 1.42, "2nd~2.9": 2.9}
# 영역별 RDF용 설정 (조성은 peak 위치엔 영향 적음 — 아는 원소로 대략)
CFG_RDF = fds.RDFConfig(composition={"Li":1,"O":1}, q_int_min=0.15, q_int_max=1.0,
                        r_min=1.0, r_max=8.0, dr=0.02, damping="lorch")
# ★ 캘리브레이션 고정(known-standard): 비정질 첫 링(FSDP)의 최근접이웃 거리를 아는 값으로 맞춤.
#   None이면 메타데이터 q_per_px 그대로 사용(가정 없음).
#   Li-음이온이 주성분이면 CALIB_R_TARGET=2.0 (Å)로 두면 RDF 첫 peak이 그 거리로 옴.
#   → 메타데이터 대비 큰 보정이 필요하면 카메라 길이/단위를 다시 확인하세요.
CALIB_R_TARGET = None     # 예: 2.0  (아는 최근접이웃 거리, Å) / None=메타데이터 신뢰
# --- 빈(진공) 위치 제외 ---
# ★ nb5와 마스크를 '똑같이' 하려면 EMPTY_ROWS/EMPTY_ROI를 nb5와 같은 값으로 두세요.
#   (지정하면 그 빈 영역의 산란 평균+3σ로 임계 = nb5와 동일 방법. None이면 자동 Otsu라 조금 다름)
MATERIAL_MASK = True      # 물질 없는(진공) 위치를 분석에서 제외
ERODE_EDGE    = 2         # 물질 마스크를 이만큼(px) 침식 → 얇은 '가장자리 상'(두께 효과) 제외, 벌크만 분석
HOT_THRESHOLD = 8.0       # 고정 hot/dead 픽셀 검출 민감도(작을수록 민감)
DENOISE_CUBE  = False     # True면 전체 큐브의 고정 bad 픽셀 수리(메모리 큼). False면 평균 패턴만 정리
EMPTY_ROWS    = 10        # 아래 N행이 빈 영역(임계 기준). nb5의 EMPTY_ROWS와 같게 맞추세요
EMPTY_ROI     = None      # (y0,y1,x0,x1)로 빈 영역 직접 지정(있으면 EMPTY_ROWS 무시)
# 거리 밴드(Å) — 각각 하나의 FC-STEM 이미지. 데이터에 맞게 조절(먼저 §3 프로파일 보고).
BANDS      = [(1.0,1.5),(1.5,2.0),(2.0,2.5),(2.5,3.0),(3.0,3.5),(3.5,4.0),(4.0,4.5),(4.5,5.0),(5.0,5.5),(5.5,6.0)]

def make_fcstem_cube(scan=(36,48), dp=(96,96), empty_rows=8, seed=0):
    '''합성: 좌=결정(스팟), 중=비정질(halo), 우=다른 비정질(halo2), 아래=빈영역.
    FC-STEM이 결정/비정질/빈영역을 구분하는지 확인용.'''
    rng=np.random.default_rng(seed); Sy,Sx=scan; H,W=dp
    yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/(2*2.0**2))
    def halo(r0,s=4.0): return np.exp(-(rr-r0)**2/(2*s**2))
    def spots(r0,n=6,s=1.6,amp=4.0):
        img=np.zeros((H,W))
        for k in range(n):
            a=2*np.pi*k/n; sx,sy=cx+r0*np.cos(a),cy+r0*np.sin(a)
            img+=amp*np.exp(-((xx-sx)**2+(yy-sy)**2)/(2*s**2))
        return img
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            if iy>=Sy-empty_rows: base=0.9*beam                      # 빈 영역(진공)
            elif ix<Sx//3:        base=beam+spots(22)+0.5*halo(22)   # 결정
            elif ix<2*Sx//3:      base=beam+1.2*halo(20)             # 비정질 A
            else:                 base=beam+1.2*halo(28)             # 비정질 B
            cube[iy,ix]=base+0.15*rng.standard_normal((H,W))
    return np.clip(cube,0,None)

if USE_SYNTHETIC:
    cube=fds.from_array(make_fcstem_cube(empty_rows=(EMPTY_ROWS or 8)), q_per_px=Q_PER_PX, name="synthetic-FCSTEM")
else:
    cube=fds.load(DM4_PATH, Q_UNIT_HINT)
    print("raw loaded shape:", cube.data.shape, "(ndim", cube.ndim, ")")
    if cube.ndim<3:
        raise ValueError(f"{cube.ndim}D — not a scan; 로더가 데이터셋 검색 후에도 2D면 직접 로드하세요.")
    if DET_BIN>1: cube=fds.bin_cube_detector(cube, DET_BIN)
scan=cube.scan_shape; dp=cube.dp_shape
QPP = cube.calibration.q_per_px or Q_PER_PX
DR  = fds.quefrency_per_px(dp[0], QPP)      # 켑스트럼 픽셀당 Å
print("cube:", cube.shape, "| scan:", scan, "| dp:", dp)
print(f"q_per_px = {QPP:.5g} 1/A/px  ->  cepstral dr = {DR:.4g} A/px,  r_max ~ {DR*(dp[0]//2):.1f} A")

# 표시용 맵 재배열(3D 스택 대비) + 저장 헬퍼
import math
def _mapshape(n):
    r=int(math.sqrt(n))
    while r>1 and n%r: r-=1
    return (r,n//r) if r>1 else (1,n)
MAP=tuple(scan) if len(scan)==2 else _mapshape(int(np.prod(scan)))
def as_map(v):
    v=np.asarray(v,float).ravel(); m=np.full(int(np.prod(MAP)),np.nan); m[:min(v.size,m.size)]=v[:m.size]; return m.reshape(MAP)
SAVE_DIR=(os.path.dirname(DM4_PATH)+"/nb6_outputs" if not USE_SYNTHETIC else "nb6_outputs")
os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,name): p=os.path.join(SAVE_DIR,name+".png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(name,header,rows):
    import csv; p=os.path.join(SAVE_DIR,name+".csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:",p)
print("outputs ->", os.path.abspath(SAVE_DIR))



## 2) median · **MAX 프로젝션** · 물질 마스크

median은 약해서 잘 안 보일 때, **MAX 프로젝션**(각 검출기 픽셀의 스캔 전체 최댓값)이 **가장 강한 신호**
(결정 스팟 등)를 모아 보여줍니다. 그리고 산란 세기로 **물질 마스크**를 만들어 **빈(진공) 위치를 분석에서
제외**합니다(흰색=분석). 이후 모든 단계는 물질 위치만 씁니다.


In [ ]:

med=fds.median_pattern(cube); mx=cube.max_dp()
if CENTER is not None: cx,cy=CENTER
else: cx,cy=fds.center_of_mass(med, threshold=0.3)

# 물질 마스크 (빈 영역 제외)
if MATERIAL_MASK and len(scan)==2:
    empty=None
    if EMPTY_ROI is not None:
        empty=np.zeros(scan,bool); y0,y1,x0,x1=EMPTY_ROI; empty[y0:y1,x0:x1]=True
    elif EMPTY_ROWS:
        empty=np.zeros(scan,bool); empty[max(0,scan[0]-EMPTY_ROWS):,:]=True
    material=fds.material_mask(cube, center=(cx,cy), empty_mask=empty)
    if ERODE_EDGE > 0:                               # 얇은 가장자리(두께 효과) 제외 → 벌크만
        try:
            from scipy.ndimage import binary_erosion
            material = binary_erosion(material, iterations=int(ERODE_EDGE))
        except Exception: pass
else:
    material=np.ones(scan,bool)
KEEP=material.ravel()
def scatter(vec, fill=np.nan):   # 물질 전용 결과를 스캔 전체로 되돌림(빈 곳=fill)
    out=np.full(KEEP.size,fill,float); out[KEEP]=np.asarray(vec,float).ravel(); return out.reshape(scan)
print(f"center: ({cx:.1f},{cy:.1f}) | material positions: {int(KEEP.sum())}/{KEEP.size} ({100*KEEP.mean():.0f}%)")

fig,ax=plt.subplots(1,3,figsize=(13,4.2))
im=ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(cx,cy,"c+",ms=10); ax[0].set_title("median (log)"); ax[0].axis("off")
im=ax[1].imshow(np.log1p(mx),cmap="magma"); ax[1].plot(cx,cy,"c+",ms=10); ax[1].set_title("MAX projection (strongest signal)"); ax[1].axis("off")
ax[2].imshow(material,cmap="gray"); ax[2].set_title("material mask (white = analyzed)"); ax[2].axis("off")
plt.tight_layout(); save(fig,"02_median_max_material"); plt.show()
np.save(os.path.join(SAVE_DIR,"02_max_projection.npy"), np.asarray(mx))



## 2b) 빔 wander 점검 (전처리)

넓은 영역에서 **직접빔이 흔들리거나 기울면**, 원시 패턴 PCA는 구조 대신 빔 움직임을 잡습니다(nb5 §4에서
확인됨). 각 위치의 **빔 중심 이동**을 매핑해 그 정도를 봅니다. 켑스트럼(§3~5)은 **병진 불변**이라 영향
없지만, **영역별 RDF(§6)** 는 민감하므로 §6에서 각 패턴을 **자기 빔 중심으로 정렬 후 평균**합니다.


In [ ]:

comx, comy = fds.center_of_mass_map(cube, normalize=True)   # 위치별 빔 이동(기하중심 대비)
comx = np.asarray(comx); comy = np.asarray(comy)
shift = np.hypot(comx, comy)
print(f"beam wander: median {np.nanmedian(shift[material]):.2f} px, "
      f"max {np.nanmax(shift[material]):.2f} px (물질 영역)")
fig,ax=plt.subplots(1,3,figsize=(14,3.6))
for a,d,t in [(ax[0],np.where(material,comx,np.nan),"beam shift x (px)"),
              (ax[1],np.where(material,comy,np.nan),"beam shift y (px)"),
              (ax[2],np.where(material,shift,np.nan),"|beam shift| (px)")]:
    im=a.imshow(d,cmap="coolwarm" if "shift x" in t or "shift y" in t else "viridis")
    a.set_title(t); a.axis("off"); plt.colorbar(im,ax=a,fraction=0.046)
fig.suptitle("beam wander across the scan (smooth gradient = beam tilt)", y=1.04)
plt.tight_layout(); save(fig,"02b_beam_wander"); plt.show()


## 2c) 노이즈(고정 hot/dead 픽셀) & **캘리브레이션**

**노이즈**: 매 프레임 같은 위치의 hot/dead 픽셀은 평균으로도 안 없어지고 log(EWPC)를 왜곡합니다. bad-pixel
맵으로 검출하고(구조는 보호), 영역 평균 패턴은 §6 RDF 전에 정리합니다. `DENOISE_CUBE=True`면 전체 큐브 수리.

**캘리브레이션(중요)**: 비정질 첫 링은 **FSDP**입니다. 여기서 두 거리를 구분해야 합니다 —

- 결정 격자면 간격 $d = 1/q$
- 비정질 **최근접이웃** $r_{nn} \approx 1.23/q$ (Ehrenfest 관계). **RDF 첫 peak도 여기로 나옵니다.**

즉 링이 $q\approx0.62$ Å⁻¹이면 RDF 첫 peak은 $1.23/0.62\approx2.0$ Å. 관측 peak이 1.5 Å 근처라면
링이 $q\approx0.82$로 잡힌 것 → **q 스케일(=q_per_px)이 ~30% 큼**을 뜻합니다. Li-음이온 최근접이웃을
아는 값(≈2.0 Å)으로 `CALIB_R_TARGET`에 넣으면 q_per_px를 그에 맞게 보정하고, 필요한 보정폭도 출력합니다
(보정폭이 크면 카메라 길이/단위 메타데이터를 재확인하세요).

In [ ]:
badmap = fds.bad_pixel_map(np.asarray(cube.max_dp(),float), hot_threshold=HOT_THRESHOLD)
print(f"bad (hot/dead) pixels: {int(badmap.sum())} ({100*badmap.mean():.2f}% of detector)")
if DENOISE_CUBE and badmap.sum()>0:
    cube = fds.repair_bad_pixels(cube, badmap); print("repaired whole cube")

# --- 첫 링(FSDP) 위치 측정 ---
mp = fds.average_pattern(cube, material)
qd, Id = fds.azimuthal_integrate(mp, (cx,cy), q_per_px=QPP)
sel = qd > CFG_RDF.q_int_min*1.5
q_ring = float(qd[sel][np.argmax(Id[sel])]) if sel.any() else np.nan
r_nn = 1.23/q_ring if np.isfinite(q_ring) and q_ring>0 else np.nan
print(f"first ring (FSDP): q = {q_ring:.3f} 1/A")
print(f"   crystallographic d = 1/q      = {1/q_ring:.2f} A")
print(f"   amorphous nn ~ 1.23/q (RDF첫peak) = {r_nn:.2f} A   (Li-anion 예상 ~2.0 A)")

# --- (선택) 아는 최근접이웃 거리로 q 캘리브레이션 ---
if CALIB_R_TARGET and np.isfinite(q_ring) and q_ring>0:
    q_target = 1.23/CALIB_R_TARGET            # nn=CALIB_R_TARGET 되려면 링이 여기 있어야
    scale = q_target/q_ring                    # q_per_px 보정 배수
    QPP = QPP*scale
    DR  = fds.quefrency_per_px(dp[0], QPP)
    cube.calibration.q_per_px = QPP
    qd, Id = fds.azimuthal_integrate(mp, (cx,cy), q_per_px=QPP)
    q_ring = q_target
    print(f"\nCALIBRATED to nn={CALIB_R_TARGET:.2f} A: q_per_px x{scale:.3f} = {QPP:.5g} 1/A/px")
    print(f"   메타데이터 대비 {abs(1-scale)*100:.0f}% 보정 -> 카메라길이/단위 재확인 권장")
print(f"q_per_px = {QPP:.5g} 1/A/px | q_max = {QPP*(dp[0]//2):.2f} 1/A -> RDF dr~{1/(QPP*(dp[0]//2)):.2f} A | cepstral dr={DR:.4g} A/px")

fig,ax=plt.subplots(1,2,figsize=(12,4.2))
ax[0].imshow(badmap,cmap="gray"); ax[0].set_title(f"bad-pixel map ({int(badmap.sum())})"); ax[0].axis("off")
ax[1].plot(qd,Id,"k-")
if np.isfinite(q_ring): ax[1].axvline(q_ring,color="r",ls="--",label=f"1st ring q={q_ring:.2f} → nn={1.23/q_ring:.2f}Å")
for lbl,d in CEPSTRAL_REF.items():
    if d>0 and 1.23/d<=qd.max(): ax[1].axvline(1.23/d,color="0.7",ls=":",lw=0.7)
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("I(q)"); ax[1].set_title("mean I(q) — calibration (dotted = 1.23/d_ref)"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"02c_denoise_calib"); plt.show()



## 3) 평균 EWPC + 켑스트럼 방사 프로파일 (원자간 거리)

전체 평균 켑스트럼과 그 방사 프로파일. **피크 = 원자간 거리(Å)**. 이게 약한 신호에서도 나오는 게 핵심.
아래 색 밴드가 §4 FC-STEM에 쓰는 거리 구간입니다. 피크가 물리적으로 타당한 Å(예: 2~4 Å)인지 확인하고,
어긋나면 `Q_UNIT_HINT`/`Q_PER_PX`(→ dr) 를 조정하세요.


In [ ]:

matflat = cube._flat_patterns()[KEEP]          # 물질 위치 패턴만
mcep=fds.ewpc_mean(matflat, offset=EWPC_OFFSET, window=WINDOW, reducer="mean", n_jobs=N_JOBS, progress=True)
r_ax, prof=fds.cepstral_radial_profile(mcep, QPP, r_min=0.4)
n=mcep.shape[0]; ext=DR*(n//2)
fig,ax=plt.subplots(1,2,figsize=(12,4.4))
im=ax[0].imshow(mcep, cmap="inferno", extent=[-ext,ext,-ext,ext],
                vmax=np.percentile(mcep,99.5)); ax[0].set_title("mean EWPC (quefrency, Å)")
ax[0].set_xlabel("r_x (Å)"); ax[0].set_ylabel("r_y (Å)"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[1].plot(r_ax, prof, "k-")
for (ri,ro),c in zip(BANDS, plt.cm.viridis(np.linspace(0,0.85,len(BANDS)))):
    ax[1].axvspan(ri,ro,color=c,alpha=0.12)
for lbl,d in CEPSTRAL_REF.items():                 # 알려진 Li 화합물 거리
    if d<=r_ax.max():
        ax[1].axvline(d,color="0.5",ls=":",lw=0.8)
        ax[1].text(d, ax[1].get_ylim()[1]*0.98, lbl.split()[0], rotation=90, fontsize=5, va="top", ha="right", color="0.4")
ax[1].set_xlabel("quefrency r (Å)"); ax[1].set_ylabel("cepstral intensity")
ax[1].set_title("radial cepstral profile (peaks = interatomic distances)")
plt.tight_layout(); save(fig,"03_mean_ewpc_profile"); plt.show()
np.save(os.path.join(SAVE_DIR,"03_mean_ewpc.npy"), mcep)
save_csv("03_cepstral_radial_profile", ["r_A","cepstral_intensity"],
         [[f"{r_ax[i]:.4f}", f"{prof[i]:.6g}"] for i in range(len(r_ax))])



## 4) FC-STEM 이미지 — 거리 밴드별 fluctuation (혼합상 매핑)

각 거리 밴드에서 켑스트럼의 정규분산 $F(R_p)$ 를 스캔에 매핑. **밝음 = 큰 fluctuation**(그 거리 범위에서
질서/스펙클이 큼 → 결정질/특정 상). 밴드를 바꾸면 다른 상이 드러납니다.


In [ ]:

maps=fds.fluctuation_multiband(cube, BANDS, QPP, offset=EWPC_OFFSET, window=WINDOW,
                               n_jobs=N_JOBS, progress=True)
maps=[np.where(material, np.asarray(m), np.nan) for m in maps]   # 빈(진공) 위치 제외
nB=len(BANDS)
fig,ax=plt.subplots(1,nB,figsize=(3.2*nB,3.4),squeeze=False)
for j,((ri,ro),m) in enumerate(zip(BANDS,maps)):
    im=ax[0][j].imshow(as_map(m), cmap="viridis"); ax[0][j].set_title(f"F: {ri}-{ro} Å", fontsize=9)
    ax[0][j].axis("off"); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
fig.suptitle("FC-STEM fluctuation images (bright = ordered in that distance band)", y=1.03)
plt.tight_layout(); save(fig,"04_fcstem_fluctuation"); plt.show()
save_csv("04_fcstem_fluctuation", ["scan_index"]+[f"F_{ri}_{ro}A" for ri,ro in BANDS],
         [[i]+[f"{np.asarray(m).ravel()[i]:.6g}" for m in maps] for i in range(np.asarray(maps[0]).size)])



## 5) 켑스트럼 프로파일 분해 (NMF k=4) — 상 분리

각 위치의 켑스트럼 방사 프로파일(원자간 거리 시그니처)을 분해. 켑스트럼은 **비음수**라 NMF가 유효합니다.
먼저 **성분 수를 데이터로 확인**(5a: elbow + 누적분산)하고, 그 수로 분해(5b)해서 **성분 피크를 알려진 Li
화합물 거리와 비교**합니다. → "K=3이 진짜인가"에 대한 객관적 근거.


In [ ]:

profs, r_p = fds.ewpc_profiles(matflat, QPP, r_min=0.4, offset=EWPC_OFFSET, window=WINDOW,
                               n_jobs=N_JOBS, progress=True)   # 물질 위치만
profs = np.clip(np.nan_to_num(profs), 0, None)                 # 비음수 보장

# --- 5a) 성분 수 확인: NMF 재구성 오차 + PCA 누적분산 vs K ---
Xn = np.linalg.norm(profs) + 1e-9
errs = []
for k in range(1, KMAX_SCAN+1):
    d = fds.decompose_profiles(profs, n_components=k, method="nmf", x=r_p)
    errs.append(float(getattr(d.model, "reconstruction_err_", np.nan)) / Xn)
pcp = fds.decompose_profiles(profs, n_components=KMAX_SCAN, method="pca", x=r_p)
cum = np.cumsum(pcp.explained_variance_ratio)
for fr in (0.95, 0.99):
    idx = np.where(cum >= fr)[0]
    if idx.size: print(f"cepstral profiles reach {int(fr*100)}% variance at K = {idx[0]+1}")
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,KMAX_SCAN+1), errs, "o-"); ax[0].set_xlabel("K"); ax[0].set_ylabel("relative recon. error")
ax[0].set_title("NMF error vs K (elbow ~ #structures)"); ax[0].axvline(K, color="r", ls=":", label=f"K={K}"); ax[0].legend()
ax[1].plot(range(1,KMAX_SCAN+1), cum, "s-"); ax[1].axhline(0.99, color="r", ls=":"); ax[1].axhline(0.95, color="orange", ls=":")
ax[1].set_ylim(0,1.02); ax[1].set_xlabel("K"); ax[1].set_ylabel("cumulative var"); ax[1].set_title("PCA cumulative variance")
plt.tight_layout(); save(fig,"05a_component_scan"); plt.show()

# --- 5b) K 성분 분해 + Li 화합물 거리 참고선 ---
METHOD = "nmf"
dp_c = fds.decompose_profiles(profs, n_components=K, method=METHOD, x=r_p)
frac = dp_c.fractions
fig,ax=plt.subplots(2,K,figsize=(3.4*K,6))
for i in range(K):
    ax[0,i].plot(r_p, dp_c.components[i], lw=1.4)
    for lbl,d in CEPSTRAL_REF.items():
        if d <= r_p.max():
            ax[0,i].axvline(d, color="0.6", ls=":", lw=0.8)
            if i==0: ax[0,i].text(d, ax[0,i].get_ylim()[1]*0.98, lbl.split()[0], rotation=90, fontsize=5, va="top", ha="right", color="0.4")
    ax[0,i].set_title(f"comp {i+1}", fontsize=9); ax[0,i].set_xlabel("r (Å)")
    im=ax[1,i].imshow(scatter(frac[:,i]), cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9); ax[1,i].axis("off")
fig.suptitle(f"cepstral {METHOD.upper()} (k={K}) — peaks vs Li-compound distances", y=1.02)
plt.tight_layout(); save(fig,"05_cepstral_profile_nmf"); plt.show()
save_csv("05_cepstral_components", ["r_A"]+[f"comp{i+1}" for i in range(K)],
         [[f"{r_p[j]:.4f}"]+[f"{dp_c.components[i][j]:.6g}" for i in range(K)] for j in range(len(r_p))])
print("참고 거리(Å):", CEPSTRAL_REF)



## 6) 켑스트럼 상(相)별 RDF — 주요 peak / 물질 확인

§5 NMF 분율의 **argmax로 각 위치를 한 상에 배정**(K개 영역) → 각 영역의 패턴을 **빔 중심 정렬 후 평균**
(`average_pattern_aligned`, wander 제거) → **RDF**. 상별로 어떤 원자간 거리(peak)가 나오는지 보고, Li
화합물 참고선과 비교해 **각 상이 어떤 물질인지** 추정합니다.


In [ ]:

labels = dp_c.fractions.argmax(1)                    # 물질 위치별 상 라벨(0..K-1)
labfull = np.full(KEEP.size, -1, int); labfull[KEEP] = labels
labmap = labfull.reshape(scan)
tgt = (dp[1]/2.0, dp[0]/2.0)                          # 정렬 목표 = 검출기 중심
beam = max(2, int(CFG_RDF.q_int_min/QPP))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
im=ax[0].imshow(np.where(labmap>=0, labmap, np.nan), cmap="tab10"); ax[0].set_title(f"phase label map (K={K})")
ax[0].axis("off"); plt.colorbar(im, ax=ax[0], fraction=0.046, ticks=range(K))
rdf_rows=[]
for k in range(K):
    region = (labmap == k)
    if region.sum() < 5: continue
    pat = fds.average_pattern_aligned(cube, region, target=tgt, threshold=0.3)
    pat = fds.clean_pattern(pat, hot_threshold=HOT_THRESHOLD)     # 고정 hot/dead 정리
    rr = fds.pattern_to_rdf(pat, QPP, CFG_RDF, center=tgt, center_beam_radius=beam)
    ax[1].plot(rr.r, rr.Gr, lw=1.4, label=f"phase {k} ({100*region.mean():.0f}%)")
    rdf_rows.append((k, rr))
for lbl,d in CEPSTRAL_REF.items():
    if d<=8: ax[1].axvline(d,color="0.6",ls=":",lw=0.8); ax[1].text(d, ax[1].get_ylim()[1]*0.98, lbl.split()[0], rotation=90, fontsize=5, va="top", ha="right", color="0.4")
ax[1].axhline(0,color="0.85",lw=.8); ax[1].set_xlim(0,8); ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("G(r)")
ax[1].set_title("per-phase RDF (aligned avg) vs Li-compound distances"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"06_phase_rdf"); plt.show()
if rdf_rows:
    r0=rdf_rows[0][1].r
    save_csv("06_phase_rdf", ["r_A"]+[f"phase{k}" for k,_ in rdf_rows],
             [[f"{r0[j]:.4f}"]+[f"{rr.Gr[j]:.6g}" for _,rr in rdf_rows] for j in range(len(r0))])
print("\nAll outputs in:", os.path.abspath(SAVE_DIR))


## 6b) 후보 화합물 지문(1st + **2nd peak**) 비교 — 어떤 Li 상인가

1st peak만으론 **LiF·Li2O·Li3N·Li2CO3가 모두 ~2.0 Å**로 겹칩니다. 구별의 핵심은 **2nd peak**과
그 **비율 r₂/r₁**(구조 지문)입니다:

| 화합물 | 구조 | 1st (양이온–음이온) | 2nd (음이온–음이온) | **r₂/r₁** |
|---|---|---|---|---|
| **LiF** | rocksalt | 2.01 (Li–F) | 2.85 (F–F) | **1.41** (√2) |
| **Li2O** | antifluorite | 2.00 (Li–O) | 3.27 (O–O) | **1.63** |
| **Li3N** | hex | 1.94 (Li–N) | 3.65 (N–N) | **1.88** |
| **Li2S** | antifluorite | 2.48 (Li–S) | 4.04 (S–S) | **1.63** |
| **Li2CO3** | 분자성 | **1.28 (C–O)** · 2.0 (Li–O) | 2.22 (O–O) | — |

판별 포인트: **① Li2S**는 1st가 2.48로 홀로 큼. **② Li2CO3**는 ~1.28 Å에 **짧은 C–O**(다른 상엔 없음).
**③ LiF vs Li2O/Li3N**는 2nd(또는 r₂/r₁)로 갈림. **r₂/r₁은 q 스케일(캘리브레이션)에 무관**하므로
캘리브레이션이 불확실해도 이 비율은 신뢰할 수 있습니다.

In [ ]:
# 후보 Li 상의 지문 거리(Å, 결정 근사). 필요시 수정.
COMPOUND_REF = {
  'LiF'   : dict(r1=2.01, r2=2.85, ratio=1.41, color='#e41a1c', marks={'Li-F':2.01,'F-F':2.85,'Li-F2':3.49}),
  'Li2O'  : dict(r1=2.00, r2=3.27, ratio=1.63, color='#377eb8', marks={'Li-O':2.00,'Li-Li':2.31,'O-O':3.27}),
  'Li3N'  : dict(r1=1.94, r2=3.65, ratio=1.88, color='#4daf4a', marks={'Li-N':1.94,'Li-N2':2.11,'N-N':3.65}),
  'Li2CO3': dict(r1=1.28, r2=2.22, ratio=1.73, color='#984ea3', marks={'C-O':1.28,'Li-O':2.00,'O-O':2.22}),
  'Li2S'  : dict(r1=2.48, r2=4.04, ratio=1.63, color='#ff7f00', marks={'Li-S':2.48,'Li-Li':2.86,'S-S':4.04}),
}
def _phase_peaks(r, G):
    '''물질 상 G(r)의 1st/2nd peak(r>1.15 Å, prominence 기준).'''
    m = r > 1.15
    pk = fds.find_peaks_1d(r[m], G[m], prominence=0.05*np.nanmax(np.abs(G[m])), distance=max(3,int(0.35/(r[1]-r[0]))))
    pk = sorted(pk, key=lambda d: d['x'])            # r 오름차순
    rs = [d['x'] for d in pk]
    r1 = rs[0] if rs else np.nan
    r2 = rs[1] if len(rs) > 1 else np.nan
    short = any(d['x'] < 1.55 for d in pk)           # ~1.28 C-O 존재?
    return r1, r2, short, pk

print('phase | r1(A)  r2(A)  r2/r1 | 짧은C-O? | best(비율) | best(절대)')
print('-'*74)
phase_fp=[]
for k, rr in rdf_rows:
    r1,r2,short,pk = _phase_peaks(rr.r, rr.Gr)
    ratio = r2/r1 if (np.isfinite(r1) and np.isfinite(r2) and r1>0) else np.nan
    # 비율 매칭(캘리브레이션 무관): r2/r1이 가장 가까운 화합물
    by_ratio = min(COMPOUND_REF, key=lambda c: abs(COMPOUND_REF[c]['ratio']-ratio)) if np.isfinite(ratio) else '-'
    # 절대 매칭: (r1,r2) 유클리드 거리
    def _d(c):
        v=COMPOUND_REF[c]; s=0; n=0
        if np.isfinite(r1): s+=(v['r1']-r1)**2; n+=1
        if np.isfinite(r2): s+=(v['r2']-r2)**2; n+=1
        return s/max(n,1)
    by_abs = min(COMPOUND_REF, key=_d) if np.isfinite(r1) else '-'
    if short: by_abs='Li2CO3?'                        # C-O 지문 우선
    phase_fp.append((k,r1,r2,ratio,short,by_ratio,by_abs))
    print(f'  {k}   | {r1:5.2f}  {r2:5.2f}  {ratio:5.2f} |   {"Y" if short else "-"}    | {by_ratio:9s} | {by_abs}')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
# (좌) 상별 RDF + 화합물 지문선(색=화합물, 실선=1st, 파선=2nd/기타)
for k, rr in rdf_rows:
    ax[0].plot(rr.r, rr.Gr, lw=1.5, label=f'phase {k}')
for c,v in COMPOUND_REF.items():
    for name,d in v['marks'].items():
        if d<=8: ax[0].axvline(d, color=v['color'], ls='-' if abs(d-v['r1'])<1e-6 else ':', lw=1.0, alpha=0.55)
    ax[0].plot([],[],color=v['color'],lw=2,label=f"{c} (r2/r1={v['ratio']})")
ax[0].axhline(0,color='0.85',lw=.8); ax[0].set_xlim(0,7); ax[0].set_xlabel('r (Å)'); ax[0].set_ylabel('G(r)')
ax[0].set_title('per-phase RDF vs compound fingerprints'); ax[0].legend(fontsize=7, ncol=2)
# (우) r2/r1 비율 비교 — 캘리브레이션 무관 지문
for c,v in COMPOUND_REF.items():
    ax[1].axhline(v['ratio'], color=v['color'], ls='--', lw=1, alpha=0.7)
    ax[1].text(len(phase_fp)-0.4, v['ratio'], c, color=v['color'], fontsize=8, va='center')
for i,(k,r1,r2,ratio,short,br,ba) in enumerate(phase_fp):
    if np.isfinite(ratio): ax[1].scatter(i, ratio, s=90, zorder=5, edgecolor='k'); ax[1].text(i, ratio+0.03, f'p{k}', ha='center', fontsize=8)
ax[1].set_xlim(-0.6,len(phase_fp)+0.3); ax[1].set_xticks(range(len(phase_fp))); ax[1].set_xticklabels([f'phase {k}' for k,*_ in phase_fp])
ax[1].set_ylabel('r2 / r1  (structure fingerprint, calibration-free)'); ax[1].set_title('measured ratio vs compounds')
plt.tight_layout(); save(fig,'06b_compound_fingerprint'); plt.show()
save_csv('06b_fingerprint', ['phase','r1_A','r2_A','r2_over_r1','short_CO','best_by_ratio','best_by_abs'],
         [[k,f'{r1:.3f}',f'{r2:.3f}',f'{ratio:.3f}',int(short),br,ba] for k,r1,r2,ratio,short,br,ba in phase_fp])



**정리** — EWPC(로그→역FFT)로 배경 제거 없이 원자간 거리 신호를 얻고, (3) 평균 프로파일, (4) 거리 밴드별
FC-STEM fluctuation 매핑, (5) 프로파일 NMF k=4 로 혼합 비정질상을 분리합니다.
- **거리 밴드(`BANDS`)**: §3 프로파일에서 상별로 두드러지는 거리 구간을 골라 넣으세요.
- 위치가 많으면 (3)(4)(5)가 병렬로 수 분 걸릴 수 있습니다(`N_JOBS`, `DET_BIN`).
- 켑스트럼 거리축은 `dr=1/(N·q_per_px)`. 프로파일 피크가 예상 Å와 다르면 `Q_UNIT_HINT`/캘리브레이션 확인.
- 결정+비정질 혼합에 특히 강합니다(결정=밴드에서 밝음). 약한 리튬화합물 신호에 RDF 대안으로 권장.
